<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/paper_summary/01_SLCP_flow_capacity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 01 — Select matched posterior and likelihood flow capacity

For every simulation budget and ML seed, this notebook trains one RQS capacity
screen per preregistered architecture on the same complete training partition.
Posterior and likelihood architectures are selected separately by held-out NLL
using the one-standard-error rule across ML seeds.  The selected architecture
is then deployed as a four-member ensemble in the following notebooks.  Every
flow runs the full six-plateau schedule and deploys its final epoch.

No official posterior samples, audit-bank rows, C2ST values, or residual-ratio
diagnostics participate in selection.  The selected nominal checkpoints are
the separate-flow baseline and are loaded unchanged by its correction stage.


In [1]:
# Google Colab setup -- safe to rerun and a no-op outside Colab.
import importlib.util
import json
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path

# Must be set before the first CUDA/PyTorch initialization in this process.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = os.environ.get("PAPER_SUMMARY_USE_DRIVE", "1") != "0"

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

def installed_version(distribution):
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return None

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        default_artifact_root = Path(
            "/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP"
        )
    else:
        default_artifact_root = Path("/content/paper_summary_SLCP_artifacts")

    repository = Path("/content/nsbi-lhc-toolkit")
    if not (repository / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, repository, env=clone_env,
        )
    else:
        run("git", "-C", repository, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", repository, "fetch", "origin", BRANCH)
        run("git", "-C", repository, "checkout", BRANCH)
        run("git", "-C", repository, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", repository, "sparse-checkout", "set", "src",
        "workshops/ml4hep_tifr_colab/paper_summary",
    )
    SOURCE_DIR = repository / "workshops" / "ml4hep_tifr_colab" / "paper_summary"

    # Colab already provides the numerical/ML stack used by these notebooks.
    # Install only the two missing modern-runtime packages normally.  In
    # particular, do not let sbibm pull its historical algorithm dependency
    # tree into the current Colab Python environment (currently Python 3.13).
    modern_requirements = []
    if installed_version("nflows") != "0.14":
        modern_requirements.append("nflows==0.14")
    if importlib.util.find_spec("pyro") is None:
        modern_requirements.append("pyro-ppl")
    if modern_requirements:
        run(sys.executable, "-m", "pip", "install", "-q", *modern_requirements)
    if installed_version("sbibm") != "1.1.0":
        # This is the same Python-3.13-safe installation used by Exercises 9
        # and 10: the SLCP task/metrics need sbibm itself, nflows, and Pyro,
        # but not sbibm's old pinned SBI/algorithm environment.
        run(
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            "sbibm==1.1.0",
        )
else:
    candidates = (
        Path.cwd(),
        Path.cwd() / "paper_summary",
        Path.cwd() / "workshops" / "ml4hep_tifr_colab" / "paper_summary",
    )
    SOURCE_DIR = next(
        (candidate.resolve() for candidate in candidates if (candidate / "config.py").is_file()),
        None,
    )
    if SOURCE_DIR is None:
        raise FileNotFoundError("Cannot locate the paper_summary source directory")
    default_artifact_root = SOURCE_DIR / "artifacts"

source_path = str(SOURCE_DIR)
if source_path not in sys.path:
    sys.path.insert(0, source_path)
os.chdir(SOURCE_DIR)

ARTIFACT_ROOT = Path(
    os.environ.get("PAPER_SUMMARY_ARTIFACT_ROOT", str(default_artifact_root))
).expanduser().resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print("Paper-summary source:", SOURCE_DIR)
print("Persistent artifact root:", ARTIFACT_ROOT)


Mounted at /content/drive
Paper-summary source: /content/nsbi-lhc-toolkit/workshops/ml4hep_tifr_colab/paper_summary
Persistent artifact root: /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP


In [2]:
from config import (
    DEFAULT_ML_SEEDS,
    PAPER_BUDGETS,
    SMOKE_BUDGETS,
    SMOKE_ML_SEEDS,
    campaign_config,
    campaign_signature,
)

PROFILE = os.environ.get("PAPER_SUMMARY_PROFILE", "PAPER").upper()
CAMPAIGN_BUDGETS = list(PAPER_BUDGETS if PROFILE == "PAPER" else SMOKE_BUDGETS)
CAMPAIGN_ML_SEEDS = list(DEFAULT_ML_SEEDS if PROFILE == "PAPER" else SMOKE_ML_SEEDS)

def execution_subset(environment_name, configured):
    raw = os.environ.get(environment_name, "").strip()
    values = list(configured) if not raw else [int(value) for value in raw.split(",")]
    unknown = set(values) - set(configured)
    if not values or unknown:
        raise ValueError(f"Invalid {environment_name}: {values}; unknown={sorted(unknown)}")
    return values

BUDGETS_TO_RUN = execution_subset("PAPER_SUMMARY_RUN_BUDGETS", CAMPAIGN_BUDGETS)
ML_SEEDS_TO_RUN = execution_subset("PAPER_SUMMARY_RUN_SEEDS", CAMPAIGN_ML_SEEDS)
LOAD_IF_AVAILABLE = True #os.environ.get("PAPER_SUMMARY_LOAD_IF_AVAILABLE", "1") != "0"

CAMPAIGN = campaign_config(profile=PROFILE)
print(json.dumps({
    "profile": PROFILE,
    "campaign_budgets": CAMPAIGN_BUDGETS,
    "campaign_ml_seeds": CAMPAIGN_ML_SEEDS,
    "budgets_to_run": BUDGETS_TO_RUN,
    "ml_seeds_to_run": ML_SEEDS_TO_RUN,
    "load_if_available": LOAD_IF_AVAILABLE,
    "campaign_signature": campaign_signature(CAMPAIGN),
}, indent=2))


{
  "profile": "PAPER",
  "campaign_budgets": [
    10000,
    100000,
    1000000
  ],
  "campaign_ml_seeds": [
    31082026,
    31082027,
    31082028
  ],
  "budgets_to_run": [
    10000,
    100000,
    1000000
  ],
  "ml_seeds_to_run": [
    31082026,
    31082027,
    31082028
  ],
  "load_if_available": true,
  "campaign_signature": "sha256-e455fa167513"
}


In [3]:
from IPython.display import display

def display_result(result):
    if hasattr(result, "style"):
        display(result.style.format(precision=4).hide(axis="index"))
    elif isinstance(result, dict):
        for name, value in result.items():
            print(f"\n{name}")
            if hasattr(value, "style"):
                display(value.style.format(precision=4).hide(axis="index"))
            else:
                display(value)
    else:
        display(result)


In [4]:
from utils import run_capacity_scan

CAPACITY_RESULT = run_capacity_scan(
    artifact_root=ARTIFACT_ROOT,
    campaign=CAMPAIGN,
    budgets_to_run=BUDGETS_TO_RUN,
    ml_seeds_to_run=ML_SEEDS_TO_RUN,
    load_if_available=LOAD_IF_AVAILABLE,
)
display_result(CAPACITY_RESULT)


Training conditional quadratic-spline flow on 900,368 rows
  epoch 01/40: train=7.4924, validation=4.5392, lr=1.0e-04
  epoch 40/40: train=0.4723, validation=0.5499, lr=1.0e-09
Saved spline flow to /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_1000000/likelihood/blocks6_width128/seed_31082028/screen.pt
Training conditional quadratic-spline flow on 900,368 rows
  epoch 01/40: train=9.1788, validation=7.7781, lr=1.0e-04
  epoch 40/40: train=2.3932, validation=2.4430, lr=1.0e-09
Saved spline flow to /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_1000000/likelihood/blocks8_width32/seed_31082026/screen.pt
Training conditional quadratic-spline flow on 900,368 rows
  epoch 01/40: train=9.0544, validation=7.6917, lr=1.0e-04
  epoch 40/40: train=2.4160, validation=2.4782, lr=1.0e-09
Saved spline flow to /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa1

architecture,budget,campaign_signature,checkpoint,hidden_features,ml_seed,n_coupling_layers,parameter_count,route,schema,selected_epoch,split_fingerprint,validation_nll,validation_nll_row_sem,validation_rows
blocks4_width32,10000,sha256-e455fa167513,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_10000/likelihood/blocks4_width32/seed_31082026/screen.pt,32,31082026,4,32144,likelihood,slcp_paper_summary_v2,40,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,23.1815,0.1462,1001
blocks4_width32,10000,sha256-e455fa167513,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_10000/likelihood/blocks4_width32/seed_31082027/screen.pt,32,31082027,4,32144,likelihood,slcp_paper_summary_v2,40,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,23.0381,0.1461,1001
blocks4_width32,10000,sha256-e455fa167513,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_10000/likelihood/blocks4_width32/seed_31082028/screen.pt,32,31082028,4,32144,likelihood,slcp_paper_summary_v2,40,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,22.9649,0.1476,1001
blocks6_width32,10000,sha256-e455fa167513,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_10000/likelihood/blocks6_width32/seed_31082026/screen.pt,32,31082026,6,48216,likelihood,slcp_paper_summary_v2,40,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,23.2715,0.1566,1001
blocks6_width32,10000,sha256-e455fa167513,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_10000/likelihood/blocks6_width32/seed_31082027/screen.pt,32,31082027,6,48216,likelihood,slcp_paper_summary_v2,40,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,23.1514,0.1569,1001
blocks6_width32,10000,sha256-e455fa167513,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_10000/likelihood/blocks6_width32/seed_31082028/screen.pt,32,31082028,6,48216,likelihood,slcp_paper_summary_v2,40,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,23.2797,0.1548,1001
blocks8_width32,10000,sha256-e455fa167513,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_10000/likelihood/blocks8_width32/seed_31082026/screen.pt,32,31082026,8,64288,likelihood,slcp_paper_summary_v2,40,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,23.3414,0.1595,1001
blocks8_width32,10000,sha256-e455fa167513,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_10000/likelihood/blocks8_width32/seed_31082027/screen.pt,32,31082027,8,64288,likelihood,slcp_paper_summary_v2,40,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,23.1778,0.1601,1001
blocks8_width32,10000,sha256-e455fa167513,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_10000/likelihood/blocks8_width32/seed_31082028/screen.pt,32,31082028,8,64288,likelihood,slcp_paper_summary_v2,40,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,23.4345,0.1600,1001
blocks4_width64,10000,sha256-e455fa167513,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/capacity/sha256-e455fa167513/budget_10000/likelihood/blocks4_width64/seed_31082026/screen.pt,64,31082026,4,96400,likelihood,slcp_paper_summary_v2,40,5fced83fd098cb5d60899b4594d903137ece7cd870d8400c18bb7e4bff8aaf11,22.6840,0.1487,1001



selection


{'schema': 'slcp_paper_summary_v2',
 'campaign_signature': 'sha256-e455fa167513',
 'selection_rule': 'smallest_parameter_count_within_one_SE_of_best_mean_validation_NLL',
 'routes': {'10000': {'likelihood': {'architecture': 'blocks8_width128',
    'n_coupling_layers': 8,
    'hidden_features': 128,
    'parameter_count': 646432,
    'validation_nll': 21.65443945105696,
    'best_validation_nll': 21.65443945105696,
    'one_se_threshold': 21.712129169782013,
    'n_seeds': 3},
   'posterior': {'architecture': 'blocks6_width128',
    'n_coupling_layers': 6,
    'hidden_features': 128,
    'parameter_count': 463629,
    'validation_nll': 9.779118700818225,
    'best_validation_nll': 9.767248536521818,
    'one_se_threshold': 9.780680656796791,
    'n_seeds': 3}},
  '100000': {'likelihood': {'architecture': 'blocks8_width128',
    'n_coupling_layers': 8,
    'hidden_features': 128,
    'parameter_count': 646432,
    'validation_nll': 17.764106892367945,
    'best_validation_nll': 17.764106


complete


True


missing_screen_runs


0